The code will have many drawbacks
1. sender, receiver, subject ignored
2. body text not formatted properly, should be tokenized and lemmatized
3. from corpus, remove features which are too rare
4. train test set splitted randomly and not stratified
5. folds not created for cross validation
6. no transformer created
7. assuming all structures are dfs
8. ensemble method not created. poor model

In [ ]:
import numpy as np
import pandas as pd
from conf.user_info import USER_EMAIL_ID

In [ ]:
df = pd.read_csv('../data/training_data.csv', sep='~', index_col=0)
condition = df['sender'] == f'{USER_EMAIL_ID}'
df.drop(df[condition].index, inplace=True)

In [ ]:
df_reindexed = df.reset_index(drop=True)
df_reindexed.index

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
import json

with open('../data/label_dict.txt', 'r') as file:
    label_str = file.read()

all_labels = json.loads(label_str.replace('\'', '\"'))
all_labels

In [ ]:
import re

label_list = [key for key in all_labels.keys() if re.match('Label_[0-9]', key)]
label_list

In [ ]:
mlb = MultiLabelBinarizer(classes=label_list)
labels_array = [list(st.split(',')) for st in df['labels']]
labels = mlb.fit_transform(labels_array)
labels

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='constant', fill_value='')
df_reindexed_imputed = pd.DataFrame(imputer.fit_transform(df_reindexed))

In [ ]:
df_reindexed_imputed

In [ ]:
import neattext as nt
import neattext.functions as nfx

corpus = df_reindexed_imputed[4].apply(nfx.remove_stopwords)

In [ ]:
corpus

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_features_body = tfidf.fit_transform(corpus).toarray()
X_features_body

In [ ]:
X = pd.DataFrame(X_features_body)
y = pd.DataFrame(labels)

print(X.shape,'\n', y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lst = [X_train, X_test, y_train, y_test]
for i in lst:
    print(i.shape, '\n')


In [ ]:
from skmultilearn.problem_transform import BinaryRelevance
from sklearn.naive_bayes import MultinomialNB

In [ ]:
binary_rel_clf = BinaryRelevance(MultinomialNB())
binary_rel_clf.fit(X_train, y_train)
br_predictions = binary_rel_clf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, br_predictions)